In [ ]:
# Install required packages
!pip install stable-baselines3[extra] gymnasium matplotlib pygame

In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO
import time

In [ ]:
env = gym.make('LunarLander-v3')

model = PPO(
    policy = "MlpPolicy",
    env = env,
    verbose = 1,
    n_steps = 1024,
    batch_size = 64,
    n_epochs = 4,
    gamma = 0.99,
    learning_rate = 2.5e-4,
    ent_coef = 0.01,
    vf_coef = 0.5,
    max_grad_norm = 0.5,
    device = "mps"
)

In [ ]:
model.learn(total_timesteps=1E6)
model_name = "RyanModelV2"
model.save(model_name)

In [ ]:
# Load your trained PPO model
model = PPO.load("RyanModelV2")  # Load from the extracted policy.pth

print("Model loaded successfully!")
print(f"Model action space: {model.action_space}")
print(f"Model observation space: {model.observation_space}")

In [ ]:
env = gym.make('LunarLander-v3', render_mode='human')

def visualize_agent_realtime(episodes=3, delay=0.01):
    """Visualize the agent playing in real-time"""
    print(f"Starting visualization with environment: {env.spec.id}")
    
    for episode in range(episodes):
        obs, info = env.reset()
        done = False
        truncated = False
        total_reward = 0
        step_count = 0
        
        print(f"\n=== Episode {episode + 1} ===")
        
        while not done and not truncated:
            # Get action from the trained model
            action, _ = model.predict(obs, deterministic=True)
            
            # Take action in environment
            obs, reward, done, truncated, info = env.step(action)
            total_reward += reward
            step_count += 1
            
            # Render the environment
            try:
                env.render()
            except Exception as e:
                print(f"Render error: {e}")
                break
            
            # Slow down for viewing
            time.sleep(delay)
            
            # Safety limit to prevent infinite loops
            if step_count > 1000:
                print("Step limit reached, ending episode")
                break
            
        print(f"Episode {episode + 1} finished with reward: {total_reward}, steps: {step_count}")
    
    env.close()
    print("Visualization complete!")

# Run the visualization now
visualize_agent_realtime(episodes=2, delay=0.05)

### Q-learning

In [ ]:
import gymnasium as gym 
import random 
import numpy as np 

In [ ]:
env = gym.make("FrozenLake-v1", map_name = "4x4", is_slippery = False, render_mode="rgb_array")

In [ ]:
env.observation_space

In [ ]:
env.action_space

In [ ]:
state_space = env.observation_space.n
action_space = env.action_space.n
print(f"State space: {state_space}")
print(f"Action space: {action_space}")

In [ ]:
def q_table(state_space, action_space):
    return np.zeros((state_space, action_space))

q = q_table(state_space, action_space)

In [ ]:
def greedy_policy(state, q_table):
    '''
    state: current state
    q_table: Q-table
    returns action with highest Q-value for the given state
    '''
    action = np.argmax(q_table[state][:])
    return action

In [ ]:
import random 

def epsilon_greedy(epislon, Q, state):
    if random.random() < epislon:
        return random.randint(0, 3)
    else:
        return greedy_policy(state, Q)
    

In [ ]:
def update_q_table(Q, state, action, reward, next_state, gamma, lr):
    Q[state][action] += lr * (reward + gamma * np.max(Q[next_state]) - Q[state][action])
    return Q

In [ ]:
n_training_steps = 100000
lr = 0.2
n_evaluations = 100
max_steps = 99 
gamma = 0.95 
epsilon_decay = 0.999
epsilon = 1.0

In [ ]:
from tqdm.auto import tqdm

def train_loop(episodes, lr, gamma, max_steps, epsilon, epsilon_decay, q_table):
    for episode in tqdm(range(episodes)):
        state, info = env.reset()
        
        for step in range(max_steps):
            action = epsilon_greedy(epsilon, q_table, state)
            next_state, reward, done, truncated, info = env.step(action) 
            
            q_table = update_q_table(q_table, state, action, reward, next_state, gamma, lr)
            state = next_state
            if step % 10 == 0:
                print(f"Episode: {episode}, Step: {step}, Epsilon: {epsilon}, Reward: {reward}")

            if done or truncated:
                break

        epsilon *= epsilon_decay
    return q_table

In [ ]:
from tqdm.auto import tqdm

def train_loop(episodes, lr, gamma, max_steps, epsilon, epsilon_decay, q_table):
    reward_list = []
    for episode in tqdm(range(episodes)):
        state, info = env.reset()

        total_reward = 0
        for step in range(max_steps):
            action = epsilon_greedy(epsilon, q_table, state)
            next_state, reward, done, truncated, info = env.step(action) 
            
            # Always update Q-table, even when episode ends
            q_table = update_q_table(q_table, state, action, reward, next_state, gamma, lr)
            state = next_state

            total_reward += reward
            if done or truncated:
                total_reward += reward 
                reward_list.append(total_reward)
                break
        
        epsilon *= epsilon_decay
    return q_table, reward_list

In [ ]:
results = train_loop(n_training_steps, lr, gamma, max_steps, epsilon, epsilon_decay, q)

In [ ]:
updated_q_table, reward_list = results
np.mean(reward_list), np.std(reward_list)

In [ ]:
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode="human")

def watch_agent_play(episodes=3):
    for episode in range(episodes):
        state, info = env.reset()
        done = False
        total_reward = 0
        
        print(f"\n=== Episode {episode + 1} ===")
        
        while not done:
            # Get action from trained Q-table (no exploration)
            action = greedy_policy(state, updated_q_table)
            
            # Take action
            next_state, reward, done, truncated, info = env.step(action)
            
            total_reward += reward
            state = next_state
            
            # The show the window 
            env.render()
            
            if done or truncated:
                break
    
    env.close()

watch_agent_play(episodes=3)

### Taxi-v3

In [ ]:
import random 
import numpy as np 
import gymnasium as gym

env = gym.make("Taxi-v3", render_mode="rgb_array")

In [ ]:
state_space = env.observation_space.n
action_space = env.action_space.n
print(f"State space: {state_space}")
print(f"Action space: {action_space}")

In [ ]:
def make_q_table(state_space, action_space):
    return np.zeros([state_space, action_space])

q_table = make_q_table(state_space, action_space)

In [ ]:
def epsilon_greedy(epsilon, q_table, state):
    if epsilon > random.random():
        return random.randint(0, action_space - 1)
    else:
        return np.argmax(q_table[state])

In [ ]:
from tqdm.auto import tqdm

def train_loop(epochs, q_table, epsilon, epsilon_decay, gamma, lr, max_steps):
    total_rewards = []

    for epoch in tqdm(range(epochs)):
        state, info = env.reset()

        for step in range(max_steps):
            move = epsilon_greedy(epsilon, q_table, state)
            next_state, reward, done, truncated, info = env.step(move)
            q_table[state, move] += lr * (reward + gamma * np.max(q_table[next_state]) - q_table[state, move])
            state = next_state
            if done or truncated:
                break
        total_rewards.append(reward)
        epsilon = max(epsilon * epsilon_decay, 0.01)
    return q_table, total_rewards

In [ ]:
updated_q_table, reward_list = train_loop(epochs = 1000000, q_table = q_table, epsilon=1, epsilon_decay=0.99, gamma = 0.9, lr = 0.1, max_steps = 1000)

In [ ]:
np.mean(reward_list), np.std(reward_list)

In [ ]:
env = gym.make("Taxi-v3", render_mode="human")

In [ ]:
def visualize_loop(epochs, q_table):
    total_rewards = []

    for epoch in tqdm(range(epochs)):
        state, info = env.reset()

        while True:
            move = np.argmax(q_table[state])
            next_state, reward, done, truncated, info = env.step(move)
            state = next_state
            env.render()
            if done or truncated:
                break
        total_rewards.append(reward)

    env.close()
    return total_rewards

In [ ]:
visualize_loop(5, updated_q_table)

### Deep Q Learning

In [ ]:
import torch 
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np 
import gymnasium as gym 
from collections import deque # Experience replay 
from tqdm.auto import tqdm 

In [36]:
env = gym.make("CartPole-v1")
state_size = env.observation_space.shape[0] # 4 variables (position, velocity, angle, angular velocity)
action_size = env.action_space.n # Right or left 
print(f"State size: {state_size}")
print(f"Action size: {action_size}")

State size: 4
Action size: 2


In [37]:
class DQN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [60]:
device = "mps" if torch.backends.mps.is_available() else "cpu" 

policy_net = DQN(state_size, 64, action_size).to(device) # Training network
target_net = DQN(state_size, 64, action_size).to(device) # Target network
target_net.load_state_dict(policy_net.state_dict()) # copy weights 
target_net.eval() 

optimizer = optim.Adam(policy_net.parameters(), lr=1E-3)
memory = deque(maxlen=100000)
batch_size = 32 
gamma = 0.99 
epsilon = 1.0 
epsilon_decay = 0.995 
epsilon_min = 0.01 

In [61]:
loss_fn = nn.MSELoss()
def optimize_model(Q_values, target_Q_values):
    optimizer.zero_grad(set_to_none=True)
    loss = loss_fn(Q_values, target_Q_values)
    loss.backward()
    optimizer.step()

def select_action(state, epsilon):
    if random.random() < epsilon:
        return env.action_space.sample()
    else:
        with torch.inference_mode():
            return policy_net(state).argmax().item()

def replay(iteration, target_update = 10):
    if len(memory) < batch_size:
        return

    if iteration % target_update == 0:
        target_net.load_state_dict(policy_net.state_dict())

    batch = random.sample(memory, batch_size) # random sample from memory 
    states, actions, rewards, next_states, dones = zip(*batch)
    states = torch.tensor(states, dtype=torch.float).to(device)
    actions = torch.tensor(actions, dtype=torch.long).to(device)
    rewards = torch.tensor(rewards, dtype=torch.float).to(device)
    next_states = torch.tensor(next_states, dtype=torch.float).to(device)
    dones = torch.tensor(dones, dtype=torch.bool).to(device)

    q_values = policy_net(states)
    q_values = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)

    with torch.no_grad():
        next_q_values = target_net(next_states) 
        target = rewards + gamma * next_q_values.max(1).values * ~dones
    
    optimize_model(q_values.to(device), target.to(device))

    return states, actions, rewards, next_states, dones

In [64]:
num_episodes = 100

for episode in range(num_episodes):
    state, info = env.reset()
    done = False
    total_reward = 0
    i = 0
    while not done:
        i += 1
        # Select action using epsilon-greedy
        action = select_action(torch.tensor(state, dtype=torch.float32).to(device), epsilon)
        
        # Take action in environment
        next_state, reward, done, _, _ = env.step(action)
        total_reward += reward
        
        # Store experience in replay memory
        memory.append((state, action, reward, next_state, done))
        
        # Move to next state
        state = next_state
        
        # Train the network
        replay(i)
    
    # Decay epsilon
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay
    
    print(f"Episode {episode}, Total Reward: {total_reward}, Epsilon: {epsilon:.2f}")

Episode 0, Total Reward: 1481.0, Epsilon: 0.15
Episode 1, Total Reward: 158.0, Epsilon: 0.15
Episode 2, Total Reward: 2913.0, Epsilon: 0.15
Episode 3, Total Reward: 621.0, Epsilon: 0.15
Episode 4, Total Reward: 3749.0, Epsilon: 0.15
Episode 5, Total Reward: 236.0, Epsilon: 0.15
Episode 6, Total Reward: 302.0, Epsilon: 0.15
Episode 7, Total Reward: 997.0, Epsilon: 0.15
Episode 8, Total Reward: 1928.0, Epsilon: 0.15
Episode 9, Total Reward: 167.0, Epsilon: 0.15
Episode 10, Total Reward: 136.0, Epsilon: 0.15
Episode 11, Total Reward: 157.0, Epsilon: 0.14
Episode 12, Total Reward: 149.0, Epsilon: 0.14
Episode 13, Total Reward: 139.0, Epsilon: 0.14
Episode 14, Total Reward: 166.0, Epsilon: 0.14
Episode 15, Total Reward: 177.0, Epsilon: 0.14
Episode 16, Total Reward: 168.0, Epsilon: 0.14
Episode 17, Total Reward: 182.0, Epsilon: 0.14
Episode 18, Total Reward: 301.0, Epsilon: 0.14
Episode 19, Total Reward: 483.0, Epsilon: 0.14
Episode 20, Total Reward: 357.0, Epsilon: 0.14
Episode 21, Total R

In [65]:
env = gym.make('CartPole-v1', render_mode='human')

In [69]:
# Evaluate
policy_net.eval()
rewards_list = []

test_episodes = 3
with torch.no_grad():
    for episode in range(test_episodes):
        state, info = env.reset()
        done = False
        total_reward = 0
        i = 0
        while not done:
            i += 1
            # Select action using epsilon-greedy
            action = policy_net(torch.tensor(state, dtype=torch.float32).to(device)).argmax().item()
            
            # Take action in environment
            next_state, reward, done, _, _ = env.step(action)
            total_reward += reward
            
            # Move to next state
            state = next_state
        
        print(f"Episode {episode + 1}: Total Reward = {total_reward}, Steps = {i}")
        rewards_list.append(total_reward)

Episode 1: Total Reward = 263.0, Steps = 263
Episode 2: Total Reward = 294.0, Steps = 294
Episode 3: Total Reward = 272.0, Steps = 272
